In [1]:
!pip install -q nltk sentence_transformers transformers
import nltk
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sentence_transformers import SentenceTransformer
from transformers import BertModel, BertTokenizer

nltk.download('treebank', quiet=True)
from nltk.corpus import treebank

/Users/larsheijnen/VSCode_S2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# each sentence is a list of (word, pos_tag) tuples
sentences = treebank.tagged_sents()

# first sentence from the treebank corpus
first_sentence = sentences[0]
print(first_sentence)

# separate the words and the corresponding POS tags
words = []
pos_order = []
for word, pos in first_sentence:
    words.append(word)
    pos_order.append(pos)

#printing sentence and order of POS tags -->  visualization of how the data looks like
print("Sentence:", " ".join(words))
print("POS Order:", " ".join(pos_order))

In [ ]:
# Create sentence length data
X_text = []
y = []
data = []

for sentence in sentences:
    tokens = []
    sentence_length = len(sentence)  # Length of the sentence
    
    for word, _ in sentence:
        tokens.append(word)
        X_text.append(word)  # Add to flattened list directly
        y.append(sentence_length)  # Add to flattened labels directly
    
    # Still keep the data structure if you need it for other purposes
    data.append((tokens, [sentence_length] * len(tokens)))

# Printing to visualize the structure
print("Sample sentence length data:")
for i, (tokens, labels) in enumerate(data[:3]):  # Show first 3 examples
    print(f"Sentence {i+1}: {' '.join(tokens)}")
    print(f"Length: {labels[0]}")  # All labels are the same for words in the same sentence
    print()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def train_rf_model(features, labels):
    """
    Train random forest regressor to predict sentence length.
    
    Parameters:
      features (np.ndarray): Feature matrix of shape (num_samples, num_features).
      labels (np.ndarray): Sentence length labels.
      
    Returns:
      model: Trained random forest model.
      metrics: Dictionary containing evaluation metrics.
    """
    # Splitting into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )
    
    # Random Forest model
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1  # Use all available cores for faster training
    )
    model.fit(X_train, y_train)
    
    # Predictions on the test set
    predictions = model.predict(X_test)
    
    # Evaluation
    metrics = {
        "mean_squared_error": mean_squared_error(y_test, predictions),
        "mean_absolute_error": mean_absolute_error(y_test, predictions),
        "r2_score": r2_score(y_test, predictions)
    }
    
    return model, metrics

In [ ]:
def cross_validate_rf_model(features, labels):
    model = RandomForestRegressor(
        n_estimators=100, 
        max_depth=20,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    r2_scores = cross_val_score(model, features, labels, cv=5, scoring='r2')
    mse_scores = -cross_val_score(model, features, labels, cv=5, scoring='neg_mean_squared_error')
    mae_scores = -cross_val_score(model, features, labels, cv=5, scoring='neg_mean_absolute_error')
    
    print("Cross-validated R² Score: ", r2_scores.mean())
    print("Cross-validated MSE: ", mse_scores.mean())
    print("Cross-validated MAE: ", mae_scores.mean())


model = SentenceTransformer('all-MiniLM-L6-v2')
X_features = model.encode(X_text)

# Train the random forest model
rf_model, rf_metrics = train_rf_model(X_features, np.array(y))
print("Random Forest Evaluation Metrics for Sentence Length Prediction:")
print(rf_metrics)

# Cross-validate the random forest model
print("\nCross-validation results:")
cross_validate_rf_model(X_features, np.array(y))

In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions(model, features, true_labels):
    predictions = model.predict(features)
    
    # Sample a subset of points for visualization
    indices = np.random.choice(range(len(true_labels)), min(1000, len(true_labels)), replace=False)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(true_labels[indices], predictions[indices], alpha=0.5)
    
    # Add perfect prediction line
    min_val = min(true_labels.min(), predictions.min())
    max_val = max(true_labels.max(), predictions.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    
    plt.xlabel('Actual Sentence Length')
    plt.ylabel('Predicted Sentence Length')
    plt.title('Word Embeddings as Probes for Sentence Length')
    plt.grid(True, alpha=0.3)
    plt.show()

# Visualize the results using the random forest model
X_test_sample = X_features[-1000:]
y_test_sample = np.array(y[-1000:])
visualize_predictions(rf_model, X_test_sample, y_test_sample)